## Forecast 2026/27 + Model comparison (table models)

Train on **league tables** through 2025/26, predict the **2026/27 table** for the confirmed squad.

Both models output a **full predicted table** (points + positions) — no match-by-match simulation.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import arviz as az
from cmdstanpy import CmdStanModel

import importlib
import helping_functions as hf
importlib.reload(hf)

from helping_functions import (
    load_matches,
    load_season_tables,
    prepare_table_stan_static,
    prepare_table_stan_hierarchical,
    build_forecast_features,
    predict_table,
    pl_2627_squad,
    STUDENT_T_NU,
    FORECAST_TRAIN_SEASONS,
)


In [ ]:
matches = load_matches()
forecast_teams = pl_2627_squad(matches)
tables = load_season_tables(matches, FORECAST_TRAIN_SEASONS)

print(f"Forecast 2026/27: {len(forecast_teams)} teams")
for i, t in enumerate(forecast_teams, 1):
    print(f"  {i:2d}. {t}")


### Model 1 — static table model


In [ ]:
stan1, team_to_idx, _, feature_stats1 = prepare_table_stan_static(
    tables, FORECAST_TRAIN_SEASONS
)
stan1["nu"] = STUDENT_T_NU
fit1 = CmdStanModel(stan_file="stan/table_static.stan").sample(
    data=stan1, seed=42, chains=4, parallel_chains=4,
    iter_warmup=1000, iter_sampling=1000, adapt_delta=0.99, show_progress=True,
)
print(fit1.diagnose())


In [ ]:
feat1 = build_forecast_features(
    matches, "2526", forecast_teams, FORECAST_TRAIN_SEASONS, feature_stats1
)
pred1 = predict_table(
    fit1, forecast_teams, team_to_idx, model="static", team_features=feat1, n_sims=800, seed=1
)
pred1 = pred1.rename(columns={"pos_median": "pos_m1", "pts_median": "pts_m1"})
pred1.head(10)


### Model 2 — hierarchical table model (2526 skills)


In [ ]:
stan2, team_to_idx2, _, season_to_idx, feature_stats2 = prepare_table_stan_hierarchical(
    tables, FORECAST_TRAIN_SEASONS
)
stan2["nu"] = STUDENT_T_NU
last_season_idx = season_to_idx["2526"]

fit2 = CmdStanModel(stan_file="stan/table_hierarchical.stan").sample(
    data=stan2, seed=42, chains=4, parallel_chains=4,
    iter_warmup=1500, iter_sampling=1500, adapt_delta=0.99, show_progress=True,
)
print(fit2.diagnose())


In [ ]:
feat2 = build_forecast_features(
    matches, "2526", forecast_teams, FORECAST_TRAIN_SEASONS, feature_stats2
)
pred2 = predict_table(
    fit2,
    forecast_teams,
    team_to_idx2,
    model="hierarchical",
    last_season_index=last_season_idx,
    team_features=feat2,
    n_sims=800,
    seed=2,
)
pred2 = pred2.rename(columns={"pos_median": "pos_m2", "pts_median": "pts_m2"})
pred2.head(10)


### Side-by-side forecast 2026/27


In [ ]:
comparison = pred1[["team","pos_m1","pts_m1"]].merge(
    pred2[["team","pos_m2","pts_m2"]], on="team"
)
comparison["pos_diff"] = comparison["pos_m1"] - comparison["pos_m2"]
comparison.sort_values("pos_m1")


In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(comparison["pos_m2"], comparison["pos_m1"])
for _, r in comparison.iterrows():
    ax.annotate(r["team"], (r["pos_m2"], r["pos_m1"]), fontsize=7, alpha=0.7)
ax.plot([1, 20], [1, 20], "k--", alpha=0.5)
ax.set_xlabel("Model 2 — median position")
ax.set_ylabel("Model 1 — median position")
ax.set_title("Forecast 2026/27: table models agree?")
ax.invert_xaxis(); ax.invert_yaxis()
plt.tight_layout(); plt.show()


### WAIC / LOO (team-season rows)


In [ ]:
idata1 = az.from_cmdstanpy(fit1)
idata2 = az.from_cmdstanpy(fit2)
loo1, loo2 = az.loo(idata1), az.loo(idata2)
waic1, waic2 = az.waic(idata1), az.waic(idata2)

print("LOO Model 1:", loo1)
print("LOO Model 2:", loo2)
print(f"Delta ELPD (M2-M1): {loo2.elpd_loo - loo1.elpd_loo:.1f}")
print(az.compare({"M1 static table": idata1, "M2 hierarchical table": idata2}, ic="loo"))


### Final assessment

- **Unit of prediction:** league **table** (points → rank), not single matches.
- **LOO** compares fit on historical **team-season points**; better ELPD ⇒ better table-level likelihood.
- For 2026/27, compare `pos_m1` vs `pos_m2`; large differences ⇒ report both forecasts.
